<a href="https://colab.research.google.com/github/MisterMirmix/LLM-Prompt-Engineering-RAG/blob/rag/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-core langchain-huggingface langchain-community faiss-cpu sentence-transformers

In [2]:
import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [4]:
import os
from langchain_core.documents import Document

In [5]:
import openai

In [6]:
from typing import List

In [7]:
from google.colab import userdata

# Инициализация путей

In [8]:
NOTEBOOK_PATH = os.getcwd()
DATA_PATH = os.path.join(NOTEBOOK_PATH, "data")

# Подготовка документа

In [9]:
# Создаем объект для считывания документа
loader = DirectoryLoader(DATA_PATH,
                         glob="**/*.txt",
                         show_progress=True,
                         loader_cls=TextLoader,
                         loader_kwargs={'encoding': 'utf-8'})

In [10]:
# Указываем длины фрагмента, на который разбиваются документы
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

In [11]:
# Считываем документы и разбиваем на фрагменты

documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
docs = text_splitter.split_documents(documents)

100%|██████████| 2/2 [00:00<00:00, 24.43it/s]


In [ ]:
# Подготовка векторного хранилища

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_db = FAISS.from_documents(docs, embedding_model)
vector_db.save_local(DATA_PATH)
# vector_db = FAISS.load_local(
#     DATA_PATH,
#     embedding_model,
#     allow_dangerous_deserialization=True
# )

#обработка retriever, context, query и получение ответа

обработка и получение контекста

In [13]:
def prepare_context(retriever_docs: List[Document]) -> Document:
    nl = '\n'
    text_context = nl.join(
        [f"Context document {num+1}: {i.page_content.replace(nl, ' ')}" for num, i in enumerate(retriever_docs)]
    )
    return text_context

In [14]:
def get_retriever_context(query_text: str, retriever) -> str:
    retriever_docs = retriever.invoke(query_text)
    final_context = prepare_context(retriever_docs)
    return final_context

подключение модели

In [15]:
YANDEX_CLOUD_FOLDER = userdata.get('YANDEX_CLOUD_FOLDER')
YANDEX_CLOUD_API_KEY = userdata.get('YANDEX_CLOUD_API_KEY')

client = openai.OpenAI(
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
    project=YANDEX_CLOUD_FOLDER
)

##обычный RAG

In [16]:
# Превращение векторного хранилища в объект получения информации retriever
# search_kwargs={"k":10} - описание количества докментов, который будет возвращать retriever

retriever = vector_db.as_retriever(search_kwargs={'k': 10})

In [17]:
# Инструкция для LLM, чтобы правильно обрабатывать информацию от пользователя

system_prompt = """You are an AI Assistant Agent for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
The answer should be in Russian. \
If you don't know the answer, say so politely. \
Keep the answer concise.

Here is the retrieved context: {context}
"""

In [18]:
query_text = "Кто такая Аксинья?"

In [19]:
context = get_retriever_context(query_text=query_text)
context

'Context document 1: —\xa0А какая нам выгода отделяться?\nContext document 2: Постояла Аксинья, а потом подошла к кровати, упала ничком и заплакала такими обильными, облегчающими и сладкими слезами, какими не плакала давным-давно.\nContext document 3: Такая погода стояла несколько дней.\nContext document 4: —\xa0Батяня, нет… рука чужая на письме.         —\xa0Читай ты, не томи!\xa0— закричала Ильинична, тяжело подкатываясь к лавке (у нее пухли ноги, ходила она, редко их переставляя, ровно на колесиках катилась).\nContext document 5: —\xa0Не десять, а четыре.         —\xa0Ну все одно, хучь и четыре,\xa0— рази мал кусок? Какой же это порядок, можно сказать? А кинь по России — таких, как ваш папаша, очень даже много.\nContext document 6: День стекал к исходу. Мирная, неописуемо сладкая баюкалась осенняя тишь. Небо, уже утратившее свой летний полновесный блеск, тускло голубело.\nContext document 7: силой вспыхнувшая страсть к Аксинье. Одна она манила его к себе, как манит путника в знобящу

In [20]:
messages = [{"role": "system", "content": system_prompt.format(context=context)}] + [{"role": "user", "content": query_text}]
assistant_response_llm = client.chat.completions.create(model=f"gpt://{YANDEX_CLOUD_FOLDER}/qwen3-235b-a22b-fp8/latest",
                                                        messages=messages,
                                                        temperature=0)

In [21]:
assistant_response_llm.choices[0].message.content

'Аксинья — персонаж, упоминаемый в нескольких контекстах: она присутствует в сцене с плачем, едет на арбе, закутав лицо платком, и появляется в разговоре с просьбой о лошадях для фельдшера. Судя по контексту, это женщина, связанная с другими героями повествования, вызывающая сильные чувства (например, у Гришеньки). Однако точное описание её личности или роли в произведении в предоставленных фрагментах отсутствует.'

## модификация HyDE

In [24]:
retriever = vector_db.as_retriever(search_kwargs={'k': 10})
system_prompt = """You are an AI Assistant Agent for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
The answer should be in Russian. \
If you don't know the answer, say so politely. \
Keep the answer concise.

Here is the retrieved context: {context}
"""
query_text = "Кто такая Аксинья?"

In [25]:
prompt_HyDE = """Given the question '{query_text}', generate a hypothetical document that directly answers this question. The document should be detailed and in-depth.
            the document size has be exactly {CHUNK_SIZE} characters. The answer should be in Russian."""
messages_HyDE = [{"role": "user", "content": prompt_HyDE.format(query_text=query_text, CHUNK_SIZE=CHUNK_SIZE)}]
response_HyDE = client.chat.completions.create(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/qwen3-235b-a22b-fp8/latest",
    messages=messages_HyDE,
    temperature=0.
)
doc_HyDE = response_HyDE.choices[0].message.content
display(doc_HyDE)

'Аксинья — персонаж русской литературы, главная героиня романа М. А. Шолохова «Тихий Дон». Полное имя — Аксинья Афанасьевна Астахова. Женщина сильная, страстная, трагическая. С детства выдана замуж за Степана Астахова, но влюбляется в Григория Мелехова. Их любовь длится годами, несмотря на разлуки, войны, браки с другими. Аксинья — образ воплощения чувственной, всепоглощающей любви. Её судьба трагична: погибает от пули во время Гражданской войны. Её образ символизирует разрушительную силу страсти и трагедию личной жизни на фоне исторических потрясений.'

In [26]:
context = get_retriever_context(query_text=doc_HyDE)
display(context)

'Context document 1: садился с ней рядом, и так подолгу глядели они в степь. Глядели до тех пор, пока истухала заря, а потом Прокофий кутал жену в зипун и на руках относил домой. Хутор терялся в догадках, подыскивая объяснение таким диковинным поступкам, бабам за разговорами поискаться некогда было. Разно гутарили и о жене Прокофия: одни утверждали, что красоты она досель невиданной, другие — наоборот.\nContext document 2: Океанские пароходы привозили не только английские и французские аэропланы, танки, пушки, пулеметы, винтовки, но и упряжных мулов, и обесцененное миром с Германией продовольствие и обмундирование. Тюки английских темно-зеленых бриджей и френчей – с вычеканенным на медных пуговицах вздыбившимся британским львом – заполнили новороссийские пакгаузы. Склады ломились от американской муки, сахара, шоколада, вин. Капиталистическая Европа, напуганная упорной живучестью большевиков, щедро слала\nContext document 3: —\xa0Возьми двух сидельцев и пойди их арестуй. Гони в правлени

In [27]:
messages = [{"role": "system", "content": system_prompt.format(context=context)}] + [{"role": "user", "content": query_text}]
assistant_response_llm = client.chat.completions.create(model=f"gpt://{YANDEX_CLOUD_FOLDER}/qwen3-235b-a22b-fp8/latest",
                                                        messages=messages,
                                                        temperature=0)
display(assistant_response_llm.choices[0].message.content)

'Аксинья — это женщина, к которой Григорий Мелехов испытывает сильные чувства. Она упоминается в контексте его тоски по дому и детям, что указывает на её важную роль в его жизни.'

## модификация ...

In [22]:
# retriever = vector_db.as_retriever(search_kwargs={'k': 10})
# system_prompt = """You are an AI Assistant Agent for question-answering tasks. \
# Use the following pieces of retrieved context to answer the question. \
# The answer should be in Russian. \
# If you don't know the answer, say so politely. \
# Keep the answer concise.

# Here is the retrieved context: {context}
# """
# query_text = "Кто такая Аксинья?"
# context = get_retriever_context(query_text=query_text)
# display(context)
# messages = [{"role": "system", "content": system_prompt.format(context=context)}] + [{"role": "user", "content": query_text}]
# assistant_response_llm = client.chat.completions.create(model=f"gpt://{YANDEX_CLOUD_FOLDER}/qwen3-235b-a22b-fp8/latest",
#                                                         messages=messages,
#                                                         temperature=0)
# display(assistant_response_llm.choices[0].message.content)